# 

In [ ]:
# проверяю, что мы, действ, на сервере
!ls

In [ ]:
!pip install -r requirements.txt

In [ ]:
## CONFIG, INCLUDE

In [ ]:

import os
import sys
import json
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from datetime import datetime
from collections import defaultdict
from dataclasses import dataclass
from typing import Dict, Any, List, Optional, Callable, Tuple

from PIL import Image, ImageDraw
from tqdm import tqdm
import albumentations as A
from torch.utils.data import Dataset, DataLoader
from pycocotools.coco import COCO
import torchvision.transforms as T

from transformers import (
    AutoImageProcessor,
    AutoModelForObjectDetection,
    TrainingArguments,
    Trainer,
    pipeline,
)

from torchmetrics.detection.mean_ap import MeanAveragePrecision
from torch.utils.tensorboard import SummaryWriter
from torch.profiler import profile, record_function, ProfilerActivity

print("✓ All imports successful")
print(f"✓ CUDA available: {torch.cuda.is_available()}")
print(f"✓ CUDA device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")



## SET UP

In [ ]:

SETTINGS = {
    'workspace_root': './mini-coco-10classes',
    'model_checkpoint': 'facebook/detr-resnet-50',
    'target_categories': ['person', 'car', 'dog', 'chair', 'pizza', 'bus', 'bottle', 'train', 'horse', 'bird'],
    'max_train_count': 500,
    'max_val_count': 100,
    'image_dimension': 512,
    'batch_sz': 4,
    'num_epochs': 5,
    'learning_rate': 1e-5,
    'l2_regularization': 1e-4,
    'eval_frequency': 100,
    'log_frequency': 50,
}

SETTINGS['train_annotations'] = os.path.join(SETTINGS['workspace_root'], 'annotations', 'instances_train2017_subset.json')
SETTINGS['val_annotations'] = os.path.join(SETTINGS['workspace_root'], 'annotations', 'instances_val2017_subset.json')
SETTINGS['train_images_dir'] = os.path.join(SETTINGS['workspace_root'], 'train2017')
SETTINGS['val_images_dir'] = os.path.join(SETTINGS['workspace_root'], 'val2017')
SETTINGS['output_artifacts'] = './detr-finetuned-10classes'
SETTINGS['tensorboard_logs'] = './runs/detr-10classes'
SETTINGS['checkpoint_storage'] = './checkpoints'

for directory in [SETTINGS['workspace_root'], SETTINGS['checkpoint_storage']]:
    os.makedirs(directory, exist_ok=True)

print(f"✓ Configuration loaded")
print(f"✓ Selected categories: {SETTINGS['target_categories']}")



## DATA PREPARATION

In [ ]:
def generate_synthetic_dataset():
    """Generate synthetic dataset with random images and annotations"""
    print(" Generating synthetic dataset structure...")
    
    os.makedirs(SETTINGS['train_images_dir'], exist_ok=True)
    os.makedirs(SETTINGS['val_images_dir'], exist_ok=True)
    
    categories = [
        {'id': i, 'name': cat_name} 
        for i, cat_name in enumerate(SETTINGS['target_categories'])
    ]
    
    train_payload = {
        'images': [],
        'annotations': [],
        'categories': categories
    }
    
    val_payload = {
        'images': [],
        'annotations': [],
        'categories': categories
    }
    
    for partition_name, num_samples, storage, img_location in [
        ('train', SETTINGS['max_train_count'], train_payload, SETTINGS['train_images_dir']),
        ('val', SETTINGS['max_val_count'], val_payload, SETTINGS['val_images_dir'])
    ]:
        for sample_id in range(num_samples):
            synthetic_image = Image.new('RGB', (640, 480), color=tuple(np.random.randint(0, 256, 3)))
            file_path = os.path.join(img_location, f'img_{sample_id:06d}.jpg')
            synthetic_image.save(file_path)
            
            storage['images'].append({
                'id': sample_id,
                'file_name': f'img_{sample_id:06d}.jpg',
                'width': 640,
                'height': 480,
                'coco_url': ''
            })
            
            num_objects = np.random.randint(1, 4)
            for obj_idx in range(num_objects):
                category_id = np.random.randint(0, len(categories))
                x_coord = np.random.randint(0, 500)
                y_coord = np.random.randint(0, 400)
                width = np.random.randint(50, 150)
                height = np.random.randint(50, 150)
                
                storage['annotations'].append({
                    'id': len(storage['annotations']),
                    'image_id': sample_id,
                    'category_id': category_id,
                    'bbox': [x_coord, y_coord, width, height],
                    'area': width * height,
                    'iscrowd': 0
                })
    
    os.makedirs(os.path.dirname(SETTINGS['train_annotations']), exist_ok=True)
    with open(SETTINGS['train_annotations'], 'w') as f:
        json.dump(train_payload, f)
    with open(SETTINGS['val_annotations'], 'w') as f:
        json.dump(val_payload, f)
    
    print(" ✓ Synthetic dataset generated")
    return True

print("\n[STEP 1] Initializing data...")
if not os.path.exists(SETTINGS['train_annotations']):
    generate_synthetic_dataset()
else:
    print(" ✓ Dataset already exists")



## LABEL MAPPINGS AND TRANSFORMATIONS

In [ ]:

def extract_class_mappings(annotations_file: str) -> Tuple[Dict, Dict, Dict]:
    """Extract and build class ID mappings from annotation file"""
    coco_api = COCO(annotations_file)
    category_ids = coco_api.getCatIds()
    category_objects = coco_api.loadCats(category_ids)
    category_objects = sorted(category_objects, key=lambda x: x['id'])
    
    idx_to_class = {idx: cat['name'] for idx, cat in enumerate(category_objects)}
    class_to_idx = {name: idx for idx, name in idx_to_class.items()}
    coco_id_mapping = {cat['id']: idx for idx, cat in enumerate(category_objects)}
    
    return idx_to_class, class_to_idx, coco_id_mapping

def convert_coco_to_standard(bbox):
    """Convert COCO (x,y,w,h) to standard (xmin,ymin,xmax,ymax) format"""
    x_start, y_start, box_width, box_height = bbox
    return [x_start, y_start, x_start + box_width, y_start + box_height]

def convert_standard_to_coco(bbox):
    """Convert standard (xmin,ymin,xmax,ymax) to COCO (x,y,w,h) format"""
    x_min, y_min, x_max, y_max = bbox
    return [x_min, y_min, x_max - x_min, y_max - y_min]

def get_training_augmentations():
    """Build training data augmentation pipeline"""
    return A.Compose([
        A.LongestMaxSize(SETTINGS['image_dimension']),
        A.PadIfNeeded(SETTINGS['image_dimension'], SETTINGS['image_dimension'], border_mode=0, value=(0, 0, 0)),
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(p=0.3),
        A.HueSaturationValue(p=0.3),
        A.Rotate(limit=10, p=0.3),
        A.RandomScale(scale_limit=0.2, p=0.3),
        A.GaussianBlur(p=0.2),
        A.GaussNoise(p=0.2),
    ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['category']))

def get_inference_augmentations():
    """Build inference preprocessing pipeline (no augmentation)"""
    return A.Compose([
        A.LongestMaxSize(SETTINGS['image_dimension']),
        A.PadIfNeeded(SETTINGS['image_dimension'], SETTINGS['image_dimension'], border_mode=0, value=(0, 0, 0)),
    ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['category']))



## DATASET CLASS

In [ ]:
class ObjectDetectionDataset(Dataset):
    """Custom dataset class for object detection tasks"""
    
    def __init__(self, image_directory: str, metadata_path: str, processor, 
                 category_mapping: Dict[int, int], preprocessing: Optional[Callable] = None):
        self.image_directory = image_directory
        self.coco_handler = COCO(metadata_path)
        self.processor = processor
        self.category_mapping = category_mapping
        self.preprocessing = preprocessing
        self.sample_ids = list(self.coco_handler.imgs.keys())
    
    def __len__(self) -> int:
        return len(self.sample_ids)
    
    def retrieve_image(self, img_id: int):
        """Load image from disk"""
        image_meta = self.coco_handler.loadImgs([img_id])[0]
        image_path = os.path.join(self.image_directory, image_meta['file_name'])
        
        if not os.path.exists(image_path):
            image = Image.new('RGB', (640, 480), color='white')
        else:
            image = Image.open(image_path).convert('RGB')
        return image
    
    def retrieve_annotations(self, img_id: int) -> Dict[str, Any]:
        """Load annotations for image"""
        ann_ids = self.coco_handler.getAnnIds(imgIds=[img_id], iscrowd=None)
        annotations_list = self.coco_handler.loadAnns(ann_ids)
        
        boxes_collection = []
        classes_collection = []
        areas_collection = []
        crowd_flags = []
        
        for annotation in annotations_list:
            bbox = annotation['bbox']
            cat_id = annotation['category_id']
            
            if cat_id not in self.category_mapping:
                continue
            
            remapped_class = self.category_mapping[cat_id]
            boxes_collection.append(bbox)
            classes_collection.append(remapped_class)
            areas_collection.append(annotation.get('area', bbox[2] * bbox[3]))
            crowd_flags.append(annotation.get('iscrowd', 0))
        
        return {
            'bboxes': boxes_collection,
            'classes': classes_collection,
            'areas': areas_collection,
            'crowds': crowd_flags,
        }
    
    def __getitem__(self, idx: int) -> Dict[str, Any]:
        """Load single sample"""
        sample_id = self.sample_ids[idx]
        image = self.retrieve_image(sample_id)
        img_w, img_h = image.size
        annotation_data = self.retrieve_annotations(sample_id)
        
        bboxes_fmt1 = annotation_data['bboxes']
        classes_list = annotation_data['classes']
        areas_list = annotation_data['areas']
        crowds_list = annotation_data['crowds']
        
        # COCO -> Standard
        bboxes_fmt2 = [convert_coco_to_standard(b) for b in bboxes_fmt1]
        
        # Apply augmentations
        image_np = np.array(image)
        
        if self.preprocessing is not None and len(bboxes_fmt2) > 0:
            aug_result = self.preprocessing(image=image_np, bboxes=bboxes_fmt2, category=classes_list)
            image_np = aug_result['image']
            bboxes_fmt2 = aug_result['bboxes']
            classes_list = aug_result['category']
        
        # Standard -> COCO
        bboxes_fmt1 = [convert_standard_to_coco(b) for b in bboxes_fmt2]
        
        # Build processor annotations
        processed_annotations = []
        for box, cls, area, crowd in zip(bboxes_fmt1, classes_list, areas_list, crowds_list):
            processed_annotations.append({
                'image_id': int(sample_id),
                'category_id': int(cls),
                'bbox': list(box),
                'area': float(area),
                'iscrowd': int(crowd),
            })
        
        # Process with image processor
        encoded = self.processor(
            images=image_np,
            annotations=processed_annotations,
            return_tensors='pt'
        )
        
        result_item = {
            'pixel_values': encoded['pixel_values'].squeeze(0),
        }
        
        result_labels = encoded['labels'][0]
        result_item['labels'] = result_labels
        
        if 'pixel_mask' in encoded:
            result_item['pixel_mask'] = encoded['pixel_mask'].squeeze(0)
        
        return result_item



## BATCH COLLATION

In [ ]:
def batch_collation_fn(batch_items):
    """Collate batch for training"""
    pixel_array = [item['pixel_values'] for item in batch_items]
    label_array = [item['labels'] for item in batch_items]
    
    padded_encoding = processor.pad(pixel_array, return_tensors='pt')
    
    collated_batch = {
        'pixel_values': padded_encoding['pixel_values'],
        'labels': label_array,
    }
    
    if 'pixel_mask' in padded_encoding:
        collated_batch['pixel_mask'] = padded_encoding['pixel_mask']
    
    return collated_batch



## METRICS COMPUTATION

In [ ]:
accumulated_metric = None
evaluation_stats = {'images': 0, 'pred_boxes': 0, 'gt_boxes': 0}

def scale_normalized_boxes(boxes: torch.Tensor, height: int, width: int) -> torch.Tensor:
    """Denormalize bounding boxes"""
    if boxes.numel() == 0:
        return boxes
    
    scale_factors = boxes.new_tensor([width, height, width, height])
    return boxes * scale_factors

def evaluate_batch(eval_prediction, finalize: bool = False):
    """Compute detection metrics"""
    global accumulated_metric, evaluation_stats
    
    if accumulated_metric is None:
        accumulated_metric = MeanAveragePrecision(box_format='xywh', class_metrics=True)
    
    loss_output, class_scores, box_predictions, _, _, class_labels = eval_prediction
    
    predicted_batch = []
    ground_truth_batch = []
    
    for predicted_class, score_logits, normalized_boxes in zip(class_labels, class_scores, box_predictions):
        original_dims = predicted_class['orig_size']
        if hasattr(original_dims, 'tolist'):
            original_dims = original_dims.tolist()
        
        if len(original_dims) == 1:
            img_height, img_width = int(original_dims[0]), int(original_dims[0])
        else:
            img_height, img_width = int(original_dims[0]), int(original_dims[1])
        
        evaluation_stats['images'] += 1
        
        # Ground truth
        gt_boxes_scaled = scale_normalized_boxes(predicted_class['boxes'], img_height, img_width)
        ground_truth_batch.append({
            'boxes': gt_boxes_scaled,
            'labels': predicted_class['class_labels'],
        })
        evaluation_stats['gt_boxes'] += gt_boxes_scaled.shape[0]
        
        # Predictions
        confidence_scores = score_logits[..., -1]
        max_scores, class_ids = confidence_scores.max(dim=-1)
        pred_boxes_scaled = scale_normalized_boxes(normalized_boxes, img_height, img_width)
        
        predicted_batch.append({
            'boxes': pred_boxes_scaled,
            'scores': max_scores,
            'labels': class_ids,
        })
        evaluation_stats['pred_boxes'] += pred_boxes_scaled.shape[0]
    
    accumulated_metric.update(predicted_batch, ground_truth_batch)
    
    if not finalize:
        return {}
    
    computed_metrics = accumulated_metric.compute()
    accumulated_metric.reset()
    
    metric_dict = {}
    for metric_name in ['map', 'map_50', 'map_75', 'mar_1', 'mar_10', 'mar_100',
                        'map_small', 'map_medium', 'map_large']:
        if metric_name in computed_metrics:
            val = computed_metrics[metric_name]
            metric_dict[metric_name] = round(val.item() if isinstance(val, torch.Tensor) else float(val), 4)
    
    return metric_dict



## MODEL INITIALIZATION

In [ ]:
# Extract class information
class_idx_mapping, idx_to_class_mapping, coco_category_mapping = extract_class_mappings(SETTINGS['train_annotations'])

print(f" Categories: {class_idx_mapping}")

# Initialize processor
processor = AutoImageProcessor.from_pretrained(SETTINGS['model_checkpoint'])

# Transformations
augmentation_pipeline = get_training_augmentations()
processing_pipeline = get_inference_augmentations()

# Datasets
training_samples = ObjectDetectionDataset(
    image_directory=SETTINGS['train_images_dir'],
    metadata_path=SETTINGS['train_annotations'],
    processor=processor,
    category_mapping=coco_category_mapping,
    preprocessing=augmentation_pipeline,
)

validation_samples = ObjectDetectionDataset(
    image_directory=SETTINGS['val_images_dir'],
    metadata_path=SETTINGS['val_annotations'],
    processor=processor,
    category_mapping=coco_category_mapping,
    preprocessing=processing_pipeline,
)

# Model
detection_model = AutoModelForObjectDetection.from_pretrained(
    SETTINGS['model_checkpoint'],
    id2label=idx_to_class_mapping,
    label2id=class_idx_mapping,
    ignore_mismatched_sizes=True,
)

print(f" ✓ Model loaded: {SETTINGS['model_checkpoint']}")
print(f" ✓ Training samples: {len(training_samples)}")
print(f" ✓ Validation samples: {len(validation_samples)}")


## TRAINING EXECUTION

In [ ]:

training_config = TrainingArguments(
    output_dir=SETTINGS['output_artifacts'],
    per_device_train_batch_size=SETTINGS['batch_sz'],
    per_device_eval_batch_size=SETTINGS['batch_sz'],
    num_train_epochs=SETTINGS['num_epochs'],
    fp16=torch.cuda.is_available(),
    save_steps=max(1, len(training_samples) // (SETTINGS['batch_sz'] * 2)),
    logging_steps=SETTINGS['log_frequency'],
    learning_rate=SETTINGS['learning_rate'],
    weight_decay=SETTINGS['l2_regularization'],
    save_total_limit=2,
    remove_unused_columns=False,
    eval_steps=SETTINGS['eval_frequency'],
    eval_strategy='steps',
    report_to=['tensorboard'],
    push_to_hub=False,
    batch_eval_metrics=True,
    no_cuda=not torch.cuda.is_available(),
    use_cpu=False,
    dataloader_pin_memory=True,
    load_best_model_at_end=True,
)

model_trainer = Trainer(
    model=detection_model,
    args=training_config,
    data_collator=batch_collation_fn,
    train_dataset=training_samples,
    eval_dataset=validation_samples,
    tokenizer=processor,
    compute_metrics=evaluate_batch,
)

# Execute training
training_output = model_trainer.train(resume_from_checkpoint=False)

print("\n ✓ Training completed")
print(f" Training loss: {training_output.training_loss:.4f}")

# Persist model
model_trainer.save_model(SETTINGS['output_artifacts'])
processor.save_pretrained(SETTINGS['output_artifacts'])

print(f" ✓ Model saved to {SETTINGS['output_artifacts']}")




## INFERENCE PROFILING

In [ ]:
def profile_inference():
    """Profile model inference performance"""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    detection_model.to(device)
    detection_model.eval()
    
    sample_batch = training_samples[0]
    pixel_input = sample_batch['pixel_values'].unsqueeze(0).to(device)
    
    activity_list = [ProfilerActivity.CPU]
    if torch.cuda.is_available():
        activity_list.append(ProfilerActivity.CUDA)
    
    with profile(
        activities=activity_list,
        record_shapes=True,
        profile_memory=True
    ) as profiler:
        with torch.no_grad():
            with record_function("inference_pass"):
                _ = detection_model(pixel_values=pixel_input)
    
    profiler.export_chrome_trace(os.path.join(SETTINGS['checkpoint_storage'], 'profile.json'))
    return profiler

profiling_result = profile_inference()
print(f" ✓ Profile exported to {os.path.join(SETTINGS['checkpoint_storage'], 'profile.json')}")



## VALIDATION AND METRICS


In [ ]:
detection_model.eval()
compute_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
detection_model.to(compute_device)

metric_calculator = MeanAveragePrecision(box_format='xywh', class_metrics=True)

with torch.no_grad():
    for batch_idx, sample_data in enumerate(tqdm(validation_samples, desc='Validating', ncols=80)):
        pixel_input = sample_data['pixel_values'].unsqueeze(0).to(compute_device)
        label_info = sample_data['labels']
        
        model_output = detection_model(pixel_values=pixel_input)
        output_logits = model_output.logits[0]
        output_boxes = model_output.pred_boxes[0]
        
        class_probabilities = output_logits[..., :-1].softmax(-1)
        max_prob, predicted_classes = class_probabilities.max(dim=-1)
        
        img_dims = label_info['orig_size']
        if hasattr(img_dims, 'tolist'):
            img_dims = img_dims.tolist()
        
        if len(img_dims) == 1:
            dim_h, dim_w = int(img_dims[0]), int(img_dims[0])
        else:
            dim_h, dim_w = int(img_dims[0]), int(img_dims[1])
        
        scale_vec = torch.tensor([dim_w, dim_h, dim_w, dim_h], device=compute_device, dtype=output_boxes.dtype)
        
        gt_absolute_boxes = label_info['boxes'].to(compute_device) * scale_vec
        gt_class_ids = label_info['class_labels'].to(compute_device)
        
        pred_absolute_boxes = output_boxes * scale_vec
        
        pred_items = [{
            'boxes': pred_absolute_boxes,
            'scores': max_prob,
            'labels': predicted_classes,
        }]
        
        target_items = [{
            'boxes': gt_absolute_boxes,
            'labels': gt_class_ids,
        }]
        
        metric_calculator.update(pred_items, target_items)

metrics_computed = metric_calculator.compute()

print(f"\n Evaluation Metrics:")
print(f" mAP (0.5-0.95): {float(metrics_computed['map']):.4f}")
print(f" mAP50: {float(metrics_computed['map_50']):.4f}")
print(f" mAP75: {float(metrics_computed['map_75']):.4f}")
print(f" mAR100: {float(metrics_computed['mar_100']):.4f}")



## INFERENCE AND VISUALIZATION


In [ ]:
def draw_bounding_boxes(source_image, detections, score_threshold=0.5):
    """Visualize object detections on image"""
    if not isinstance(source_image, np.ndarray):
        img_array = np.array(source_image)
    else:
        img_array = source_image
    
    fig, axis = plt.subplots(figsize=(8, 8))
    axis.imshow(img_array)
    axis.axis('off')
    
    for detection in detections:
        if detection['score'] < score_threshold:
            continue
        
        bbox = detection['box']
        x1, y1, x2, y2 = bbox['xmin'], bbox['ymin'], bbox['xmax'], bbox['ymax']
        
        box_width, box_height = x2 - x1, y2 - y1
        rectangle = patches.Rectangle((x1, y1), box_width, box_height, linewidth=2, edgecolor='lime', facecolor='none')
        axis.add_patch(rectangle)
        
        label_text = detection.get('label', 'unknown')
        score_val = detection['score']
        axis.text(x1, max(y1 - 2, 0), f'{label_text} {score_val:.2f}', fontsize=8, color='yellow',
                 bbox=dict(facecolor='black', alpha=0.5, pad=1))
    
    plt.tight_layout()
    return fig

# Load detection pipeline
inference_pipeline = pipeline(task='object-detection', model=detection_model, image_processor=processor,
                              device=0 if torch.cuda.is_available() else -1)

# Visualize samples
import random
sample_count = 3
sample_indices = random.sample(range(len(validation_samples)), k=min(sample_count, len(validation_samples)))

for idx in sample_indices:
    image_id = validation_samples.sample_ids[idx]
    orig_image = validation_samples.retrieve_image(image_id)
    
    predictions = inference_pipeline(orig_image)
    print(f"\nImage ID {image_id}: {len(predictions)} detections")
    
    visualization = draw_bounding_boxes(orig_image, predictions, score_threshold=0.3)
    plt.savefig(os.path.join(SETTINGS['checkpoint_storage'], f'detect_{image_id}.png'), dpi=100, bbox_inches='tight')
    plt.close(visualization)

print(f" ✓ Visualizations saved to {SETTINGS['checkpoint_storage']}")



## ERROR ANALYSIS


In [ ]:
def perform_error_analysis():
    """Analyze detection errors"""
    
    error_collection = {
        'incorrect_positives': [],
        'missed_detections': [],
        'uncertain_predictions': [],
    }
    
    detection_model.eval()
    analysis_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    with torch.no_grad():
        for batch_index, sample_data in enumerate(tqdm(validation_samples[:20], desc='Error Analysis')):
            pixel_input = sample_data['pixel_values'].unsqueeze(0).to(analysis_device)
            label_data = sample_data['labels']
            
            model_result = detection_model(pixel_values=pixel_input)
            output_logits = model_result.logits[0]
            output_boxes = model_result.pred_boxes[0]
            
            prob_distribution = output_logits[..., :-1].softmax(-1)
            confidence_vals, predicted_ids = prob_distribution.max(dim=-1)
            
            true_class_list = label_data['class_labels'].cpu().numpy()
            pred_class_list = predicted_ids.cpu().numpy()
            conf_list = confidence_vals.cpu().numpy()
            
            # Incorrect positives
            for j, (pred_id, conf_score) in enumerate(zip(pred_class_list, conf_list)):
                if conf_score < 0.3:
                    continue
                
                if len(true_class_list) == 0:
                    error_collection['incorrect_positives'].append({
                        'sample_idx': batch_index,
                        'class': pred_id,
                        'confidence': float(conf_score),
                    })
                else:
                    class_exists = np.any(true_class_list == pred_id)
                    if not class_exists:
                        error_collection['incorrect_positives'].append({
                            'sample_idx': batch_index,
                            'predicted_class': pred_id,
                            'actual_classes': list(true_class_list),
                            'confidence': float(conf_score),
                        })
            
            # Missed detections
            for true_id in true_class_list:
                pred_exists = np.any(pred_class_list == true_id)
                if not pred_exists:
                    error_collection['missed_detections'].append({
                        'sample_idx': batch_index,
                        'class': true_id,
                    })
            
            # Uncertain predictions
            for j, (pred_id, conf_score) in enumerate(zip(pred_class_list, conf_list)):
                if 0.3 <= conf_score < 0.5:
                    error_collection['uncertain_predictions'].append({
                        'sample_idx': batch_index,
                        'class': pred_id,
                        'confidence': float(conf_score),
                    })
    
    return error_collection

error_analysis = perform_error_analysis()

print(f"\n Error Summary:")
print(f" Incorrect positives: {len(error_analysis['incorrect_positives'])}")
print(f" Missed detections: {len(error_analysis['missed_detections'])}")
print(f" Uncertain predictions: {len(error_analysis['uncertain_predictions'])}")



## RESULTS PERSISTENCE


In [ ]:
final_results = {
    'parameters': SETTINGS,
    'class_mappings': {
        'idx_to_class': idx_to_class_mapping,
        'class_to_idx': class_idx_mapping,
    },
    'performance_metrics': {
        'mean_average_precision': float(metrics_computed['map']),
        'map_at_50': float(metrics_computed['map_50']),
        'map_at_75': float(metrics_computed['map_75']),
        'mean_avg_recall': float(metrics_computed['mar_100']),
    },
    'error_statistics': {
        'false_positive_count': len(error_analysis['incorrect_positives']),
        'false_negative_count': len(error_analysis['missed_detections']),
        'low_confidence_count': len(error_analysis['uncertain_predictions']),
    },
    'data_counts': {
        'training_set_size': len(training_samples),
        'validation_set_size': len(validation_samples),
    },
}

with open(os.path.join(SETTINGS['checkpoint_storage'], 'results.json'), 'w') as f:
    json.dump(final_results, f, indent=2)

print(f" ✓ Results persisted")

print("\n✓ PIPELINE EXECUTION COMPLETED")
print(f"\n📊 Results location: {SETTINGS['checkpoint_storage']}")
print(f"📁 Model location: {SETTINGS['output_artifacts']}")
print(f"📈 TensorBoard: tensorboard --logdir {SETTINGS['tensorboard_logs']}")
